# Legal AI Agent — Colab GPU Embedding Pipeline (v2 — tối ưu tốc độ)

**Mục tiêu:** Embed ~89,000 văn bản pháp luật VN (~3.3–4 triệu chunks) trên GPU Colab.

**Tối ưu so với v1** (notebook cũ ước tính 9–14h và ghi thẳng Drive → vừa chậm vừa tràn 15GB):
1. ⚡ **Vectorstore ghi vào SSD local** (`/content/vectorstore`) — ghi SQLite qua Drive FUSE chậm hơn nhiều lần
2. ⚡ **FP16 trên GPU** (Embedder tự bật) — ~2× tốc độ trên T4/A100
3. ⚡ **Global batching** trong `ingest.py` — gom chunks nhiều file, GPU luôn chạy batch đầy
4. 📦 **Export cuối**: nén + copy về Drive nếu vừa, hoặc upload HuggingFace dataset (khuyến nghị — store ~25GB+ KHÔNG vừa Drive free 15GB)

| Bước | Thời gian ước tính (T4) |
|------|-------------------------|
| Download data (~2.5 GB) | 35–50 phút (lần đầu) |
| Ingest + Embed | **~1.5–2.5 giờ** |
| Build BM25 | 10–15 phút |
| Export kết quả | 20–40 phút |
| **Tổng** | **~3–4 giờ** |

> ⚠️ Colab free giới hạn 12h/session. Nếu bị ngắt: chạy lại Cell 7 → Cell 8 với `INGEST_FLAG = '--skip-existing'`.

## Cell 1 — Kiểm tra GPU & Disk

In [8]:
# ── Notebook này CHỈ chạy trên Google Colab ─────────────────────────────────
try:
    import google.colab  # noqa
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if not IS_COLAB:
    raise SystemExit(
        "
❌ Notebook này chỉ chạy trên GOOGLE COLAB (cần GPU + /content + Drive).
"
        "   KHÔNG chạy được trong VS Code / Jupyter local.

"
        "   Cách mở đúng:
"
        "   1. Vào https://colab.research.google.com
"
        "   2. File → Open notebook → tab GitHub
"
        "   3. Nhập: HoangNhatTR/ProjectGenAI_2 → chọn notebooks/colab_embedding.ipynb
"
        "   4. Runtime → Change runtime type → T4 GPU
"
        "   5. Thêm Secrets (🔑): HF_TOKEN, KIEAI_API_KEY
"
    )

import torch, os, time

# GPU check
cuda_ok = torch.cuda.is_available()
if cuda_ok:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✓ GPU  : {gpu_name} ({vram_gb:.1f} GB VRAM)')
    print(f'  batch_size sẽ dùng: 64 (T4) hoặc 128 (A100)')
    BATCH_SIZE = 128 if 'A100' in gpu_name else 64
else:
    print('⚠  KHÔNG CÓ GPU! → Runtime → Change runtime type → T4 GPU')
    BATCH_SIZE = 16

# Disk check
import shutil
total, used, free = shutil.disk_usage('/content')
print(f'\n💾 Disk: {free//1e9:.0f} GB free / {total//1e9:.0f} GB total')
if free < 15e9:
    print('⚠  Cần ít nhất 15 GB free. Xóa bớt file hoặc dùng Runtime mới.')
else:
    print('✓ Disk space OK')

print(f'\nBATCH_SIZE = {BATCH_SIZE}')

## Cell 2 — Mount Google Drive

In [9]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/LegalAI_vectorstore'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
os.makedirs(f'{DRIVE_OUTPUT}/chroma',  exist_ok=True)
print(f'✓ Drive mounted')
print(f'  Output: {DRIVE_OUTPUT}')
!ls -lh /content/drive/MyDrive/LegalAI_vectorstore/ 2>/dev/null || echo '  (thư mục trống — lần đầu chạy)'

## Cell 3 — Clone repo & cài dependencies

In [10]:
os.chdir('/content')

# Clone hoặc pull nếu đã có
if os.path.exists('/content/ProjectGenAI_2/.git'):
    print('Repo đã có, pulling latest...')
    os.chdir('/content/ProjectGenAI_2')
    !git pull origin main
else:
    !git clone https://github.com/HoangNhatTR/ProjectGenAI_2.git
    os.chdir('/content/ProjectGenAI_2')

print('\nInstalling dependencies...')
!pip install -q -r requirements.txt
print('✓ Done')

## Cell 4 — Cấu hình .env

> ✏️ Điền API keys của bạn vào đây trước khi chạy.

In [ ]:
# ── Điền API keys của bạn (KHÔNG commit key thật vào notebook!) ─────────────
# Khuyến nghị: dùng Colab Secrets (🔑 icon bên trái) thay vì gõ trực tiếp.
try:
    from google.colab import userdata
    KIEAI_API_KEY   = userdata.get('KIEAI_API_KEY')   or ""
    ROUTER9_API_KEY = userdata.get('ROUTER9_API_KEY') or ""
    GEMINI_API_KEY  = userdata.get('GEMINI_API_KEY')  or ""
except Exception:
    KIEAI_API_KEY   = ""   # ← hoặc điền tay ở đây (nhớ xóa trước khi commit)
    ROUTER9_API_KEY = ""
    GEMINI_API_KEY  = ""
# ──────────────────────────────────────────────────────────────────────────

# ⚡ v2: Vectorstore ghi vào SSD LOCAL — KHÔNG ghi thẳng Drive
#    (Drive FUSE chậm + store ~25GB sẽ tràn Drive free 15GB giữa chừng)
#    Kết quả được export ở Cell 11.
VSTORE_DIR = '/content/vectorstore'

env_content = f"""# Auto-generated for Colab — {time.strftime('%Y-%m-%d %H:%M')}

# LLM
LLM_PROVIDER=kieai
LLM_MODEL=deepseek-chat
ROUTER9_API_KEY={ROUTER9_API_KEY}
ROUTER9_BASE_URL=http://localhost:20128/v1
ROUTER9_MODEL=cc/claude-haiku-4-5-20251001
KIEAI_API_KEY={KIEAI_API_KEY}
KIEAI_BASE_URL=https://kieai.erweima.ai/api/v1
GEMINI_API_KEY={GEMINI_API_KEY}

# Embedding (Embedder tự bật CUDA + FP16 + batch size phù hợp)
EMBEDDING_MODEL=BAAI/bge-m3

# Vectorstore → SSD local (export về Drive/HF ở cuối)
VECTORSTORE_DIR={VSTORE_DIR}
COLLECTION_NAME=legal_docs

# Chunking
CHUNK_SIZE=600
CHUNK_OVERLAP=80
TOP_K=5

# Parent-Child (Điều → Khoản → Điểm)
USE_PARENT_CHILD=true
PARENT_STORE_PATH=/content/ProjectGenAI_2/data/processed/parent_store.db

# HyDE — tắt trong lúc ingest
USE_HYDE=false

# Global batching: gom đủ N chunks rồi mới embed (tăng GPU utilization)
EMBED_BUFFER_CHUNKS=1024
"""

with open('/content/ProjectGenAI_2/.env', 'w') as f:
    f.write(env_content)

os.makedirs('/content/ProjectGenAI_2/data/processed', exist_ok=True)
os.makedirs(VSTORE_DIR, exist_ok=True)
os.environ['VECTORSTORE_DIR'] = VSTORE_DIR

print('✓ .env created (vectorstore → SSD local)')
!grep -v 'API_KEY' /content/ProjectGenAI_2/.env | grep -v '^$'

## Cell 5 — Download data từ HuggingFace (~35–50 phút)

> Lần đầu: tải 88k+ VB từ thuvienphapluat.vn. Lần sau: resume tự động.

In [12]:
os.chdir('/content/ProjectGenAI_2')
!df -h /content

print('\n⏳ Downloading data (~88k văn bản, ~3.5 GB)...')
t0 = time.time()
# --resume: tự bỏ qua các VB đã tải trước đó
!python -m scripts.load_hf_dataset --resume
print(f'\n⏱ Tải xong sau {(time.time()-t0)/60:.0f} phút')

## Cell 6 — Kiểm tra data

In [13]:
from pathlib import Path

raw_dir = Path('/content/ProjectGenAI_2/data/raw')
grand_total = 0
grand_mb    = 0

for folder in sorted(raw_dir.iterdir()):
    if not folder.is_dir():
        continue
    txts = list(folder.rglob('*.txt'))
    if not txts:
        continue
    mb = sum(f.stat().st_size for f in txts) / 1e6
    print(f'  {folder.name:25s}: {len(txts):7,} files | {mb:6.0f} MB')
    grand_total += len(txts)
    grand_mb    += mb

root_txts = list(raw_dir.glob('*.txt'))
print(f'  {"(root)".ljust(25)}: {len(root_txts):7,} files')
grand_total += len(root_txts)

print(f'\n  TỔNG : {grand_total:,} files | {grand_mb:.0f} MB ({grand_mb/1024:.1f} GB)')

if grand_total < 100_000:
    print('⚠  Ít hơn 100k files — data chưa đủ. Chạy lại Cell 5.')
else:
    print('✓ Data đầy đủ, sẵn sàng ingest')

!df -h /content

## Cell 7 — Patch Embedder cho GPU

In [ ]:
import sys
sys.path.insert(0, '/content/ProjectGenAI_2')

from dotenv import load_dotenv
load_dotenv('/content/ProjectGenAI_2/.env')

# v2: KHÔNG cần monkey-patch nữa — Embedder tự detect CUDA, bật FP16,
# và chọn batch_size (64 T4 / 128 A100-L4). Chỉ cần pre-load để kiểm tra.
print('Pre-loading bge-m3 (~1 GB download)...')
from src import config
from src.embedding import Embedder

emb = Embedder(config.EMBEDDING_MODEL)
test_emb = emb.encode(['test'])
print(f'✓ Model loaded, embedding dim = {len(test_emb[0])}')

# Benchmark nhanh 256 chunks giả để ước tính throughput
import time as _t
fake = ['Điều 1. Phạm vi điều chỉnh. Nghị định này quy định về xử phạt vi phạm hành chính ' * 8] * 256
t0 = _t.time()
emb.encode(fake)
dt = _t.time() - t0
print(f'⚡ Benchmark: {256/dt:.0f} chunks/s → ước tính 3.3M chunks ≈ {3_300_000/(256/dt)/3600:.1f} giờ embed thuần')

## Cell 8 — Ingest (Embedding → Vectorstore)

> ⚡ **Bước chính — ước tính 1.5–2.5 giờ trên T4 (FP16 + global batching + SSD local).**
>
> Nếu Colab bị ngắt giữa chừng: chạy lại Cell 7 → Cell 8 với `INGEST_FLAG = '--skip-existing'`.
>
> **Lần đầu:** dùng `--reset` để tạo mới với chunking parent-child + point-level.
>
> Log in tiến độ mỗi 200 docs (docs, chunks, chunks/s, phút) — không còn ngập log per-file.

In [15]:
# Chạy cell này trước Cell 8 để pull fix
os.chdir('/content/ProjectGenAI_2')
!git pull origin main
print("✓ Fix đã được pull")

# Kiểm tra data có đủ không
from pathlib import Path
n = len(list(Path('data/raw').rglob('*.txt')))
print(f"Tổng .txt files (rglob): {n:,}")


In [ ]:
os.chdir('/content/ProjectGenAI_2')
# v2: vectorstore ở SSD local — KHÔNG phải Drive
os.environ['VECTORSTORE_DIR'] = '/content/vectorstore'
os.makedirs(os.environ['VECTORSTORE_DIR'], exist_ok=True)

# ── Chọn mode ─────────────────────────────────────────────────────────────
# Lần đầu chạy:       INGEST_FLAG = '--reset'
# Resume sau timeout: INGEST_FLAG = '--skip-existing'
INGEST_FLAG = '--reset'  # <── ĐỔI THÀNH '--skip-existing' khi resume
# ──────────────────────────────────────────────────────────────────────────

print(f'🚀 Ingest mode: {INGEST_FLAG}')
print(f'   Vectorstore  → {os.environ["VECTORSTORE_DIR"]}  (SSD local)')
print(f'   Parent-Child → data/processed/parent_store.db')
print('   (FP16 + global batching 1024 chunks + chunk Điều → Khoản → Điểm)')
print()

t0 = time.time()
!python -m scripts.ingest {INGEST_FLAG}
elapsed = (time.time() - t0) / 3600
print(f'\n✓ Ingest xong sau {elapsed:.1f} giờ')

## Cell 9 — Lưu Parent Store về Drive

In [ ]:
import shutil

src_db = '/content/ProjectGenAI_2/data/processed/parent_store.db'
dst_db = '/content/drive/MyDrive/LegalAI_vectorstore/parent_store.db'

if os.path.exists(src_db):
    shutil.copy2(src_db, dst_db)
    mb = os.path.getsize(dst_db) / 1e6
    print(f'✓ parent_store.db → Drive ({mb:.0f} MB)')
else:
    print('⚠ parent_store.db không tìm thấy!')
    print('  Kiểm tra: USE_PARENT_CHILD=true trong .env')

## Cell 10 — Build BM25 index

In [ ]:
os.chdir('/content/ProjectGenAI_2')
print('⏳ Building BM25 index từ vectorstore...')
t0 = time.time()
!python -m scripts.build_bm25
print(f'⏱ BM25 done: {(time.time()-t0)/60:.0f} phút')

# Copy BM25 về Drive
bm25_src = Path('/content/ProjectGenAI_2/data/bm25')
bm25_dst = '/content/drive/MyDrive/LegalAI_vectorstore/bm25'
if bm25_src.exists():
    shutil.copytree(str(bm25_src), bm25_dst, dirs_exist_ok=True)
    mb = sum(f.stat().st_size for f in Path(bm25_dst).rglob('*') if f.is_file()) / 1e6
    print(f'✓ BM25 index → Drive ({mb:.0f} MB)')
else:
    print('⚠ BM25 chưa được tạo')

## Cell 11 — Export vectorstore (Drive hoặc HuggingFace)

> ⚠️ **Quan trọng:** store đầy đủ ~25 GB+ — **KHÔNG vừa Google Drive free (15 GB)**.
>
> - **Option A (khuyến nghị):** upload lên **HuggingFace dataset** (private, free) — cần `HF_TOKEN` trong Colab Secrets.
> - **Option B:** nếu store nhỏ (chạy 1 phần corpus) và Drive còn chỗ → tự động nén + copy về Drive.

In [ ]:
import shutil
from pathlib import Path

VSTORE = Path('/content/vectorstore')
size_gb = sum(f.stat().st_size for f in VSTORE.rglob('*') if f.is_file()) / 1e9
print(f'📦 Vectorstore size: {size_gb:.1f} GB')

# ── Option A: HuggingFace dataset (KHUYẾN NGHỊ khi store > 10 GB) ───────────
# Tạo token (write) tại https://huggingface.co/settings/tokens
# rồi lưu vào Colab Secrets với tên HF_TOKEN
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN') or ''
except Exception:
    HF_TOKEN = ''
HF_REPO = 'HoangNhat1304/legalai-vectorstore'

if HF_TOKEN and 'HoangNhat1304' not in HF_REPO:
    print(f'⬆ Uploading lên HF dataset: {HF_REPO} (private)...')
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_REPO, repo_type='dataset', private=True, exist_ok=True)
    api.upload_folder(folder_path=str(VSTORE), repo_id=HF_REPO,
                      repo_type='dataset', path_in_repo='chroma')
    api.upload_file(path_or_fileobj='/content/ProjectGenAI_2/data/processed/parent_store.db',
                    repo_id=HF_REPO, repo_type='dataset', path_in_repo='parent_store.db')
    bm25_idx = Path('/content/ProjectGenAI_2/data/bm25/index.json')
    if bm25_idx.exists():
        api.upload_file(path_or_fileobj=str(bm25_idx), repo_id=HF_REPO,
                        repo_type='dataset', path_in_repo='bm25/index.json')
    print('✓ Upload xong. Tải về máy bằng: hf download --repo-type dataset ' + HF_REPO)

# ── Option B: Google Drive (chỉ khi store đủ nhỏ) ───────────────────────────
else:
    _, _, drive_free = shutil.disk_usage('/content/drive')
    print(f'Drive free: {drive_free/1e9:.1f} GB — cần ~{size_gb*1.05:.1f} GB')
    if drive_free > size_gb * 1.1e9:
        print('⬆ Nén thành 1 file tar rồi copy về Drive (nhanh hơn copy hàng nghìn file nhỏ)...')
        !tar -cf /content/vectorstore.tar -C /content vectorstore
        shutil.copy2('/content/vectorstore.tar',
                     '/content/drive/MyDrive/LegalAI_vectorstore/vectorstore.tar')
        print('✓ Đã copy vectorstore.tar về Drive')
        print('  Trên máy: tải về rồi `tar -xf vectorstore.tar`')
    else:
        print('❌ Drive KHÔNG đủ chỗ. Hai lựa chọn:')
        print('   1. Set HF_TOKEN trong Colab Secrets + sửa HF_REPO ở trên → chạy lại cell (khuyến nghị)')
        print('   2. Tải trực tiếp: files.download() từng phần (chậm, dễ đứt với file lớn)')

## Cell 12 — Verify kết quả

In [ ]:
from src import config
from src.vectorstore import VectorStore
from src.parent_store import ParentStore
from pathlib import Path
import os

print('=== VECTORSTORE (local SSD) ===')
store = VectorStore(config.VECTORSTORE_DIR, config.COLLECTION_NAME)
n_chunks = store.count()
print(f'  Chunks : {n_chunks:,}')

# Sample chunk để kiểm tra parent_id và point
sample = list(store.iter_all_chunks(batch_size=200))[:200]
n_with_point  = sum(1 for c in sample if c.point)
n_with_parent = sum(1 for c in sample if c.parent_id)
print(f'  Có point (Điểm a/b/c): {n_with_point}/200 mẫu {"✓" if n_with_point > 0 else "⚠ = 0 (kiểm tra lại)"}')
print(f'  Có parent_id         : {n_with_parent}/200 mẫu {"✓" if n_with_parent > 0 else "⚠ = 0 (kiểm tra lại)"}')

print('
=== PARENT STORE ===')
ps_path = '/content/ProjectGenAI_2/data/processed/parent_store.db'
if os.path.exists(ps_path):
    ps = ParentStore(Path(ps_path))
    print(f'  Parents: {ps.count():,}')
else:
    print('  ⚠ Không tìm thấy')

print('
=== BM25 ===')
bm25_path = Path('/content/ProjectGenAI_2/data/bm25')
if bm25_path.exists():
    mb = sum(f.stat().st_size for f in bm25_path.rglob('*') if f.is_file()) / 1e6
    print(f'  Size: {mb:.0f} MB ✓')
else:
    print('  ⚠ Không tìm thấy')

## Cell 13 — Test RAG query

## Hướng dẫn deploy lên server / máy local

Sau khi notebook chạy xong, kết quả nằm ở **HuggingFace dataset** (Option A) hoặc **Drive** (Option B):

```
chroma/            ← Vectorstore (~25 GB)
parent_store.db    ← Parent chunks (~1-2 GB)
bm25/index.json    ← BM25 index (~30-50 MB)
```

### Trên server / máy local:
```bash
git clone https://github.com/HoangNhatTR/ProjectGenAI_2.git && cd ProjectGenAI_2
pip install -r requirements.txt

# Option A — tải từ HuggingFace (khuyến nghị):
pip install -U huggingface_hub
hf download HoangNhat1304/legalai-vectorstore --repo-type dataset --local-dir ./data/hf_vectorstore
# → ./data/hf_vectorstore/{chroma, parent_store.db, bm25/}

# Option B — từ Drive: tải vectorstore.tar về rồi:
tar -xf vectorstore.tar   # → thư mục vectorstore/

# Cấu hình .env:
cp .env.example .env
# Sửa: VECTORSTORE_DIR=./data/hf_vectorstore/chroma   (hoặc ./vectorstore)
#      PARENT_STORE_PATH=./data/hf_vectorstore/parent_store.db
#      KIEAI_API_KEY=...
#      HF_TOKEN=...   (nếu dataset private)
# Copy bm25/index.json → data/bm25/index.json

# Chạy API
uvicorn api:app --host 0.0.0.0 --port 8000
```